In [1]:
!pip install qdrant-client fastembed -q
!pip install google-generativeai -q -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.6 MB/s eta 0:00:00


In [10]:
import json
import collections
import pandas as pd

from typing import List
from google import genai
from google.colab import userdata
from google.api_core import exceptions
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

### Load Documents and ground truth data

In [ ]:
with open('data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [11]:
questions = pd.read_csv('ground_truth.csv')

### Build Qdrant collection

In [4]:
client = QdrantClient(":memory:")

In [5]:
def collection_exists(collection_name: str) -> bool:

    try:
        existing_collections = [col.name for col in client.get_collections().collections]
        return collection_name in existing_collections

    except:
        return False

In [6]:
def build_collection(name: str, vector_config: dict = None, sparse_vector_config: dict = None):

    try:
        vector_config = vector_config or {}
        sparse_vector_config = sparse_vector_config or {}

        exists = collection_exists(name)

        if exists:
            print(f"Collection '{name}' already exists.")
            return

        client.create_collection(
            collection_name = name,
            vectors_config = vector_config,
            sparse_vectors_config = sparse_vector_config
        )

        print(f"Qdrant collection '{name}' created.")

    except Exception as e:
        print(f"Failed to create collection '{name}': {e}")

In [7]:
def populate_collection(name: str, models_names: dict, documents: List[dict]):

    try:
        exists = collection_exists(name)

        if not exists:
            print(f"Collection '{name}' does not exists.")
            return

        points = []

        for record in documents:

            text_embed = f"{record['term']}: {record['definition']} {record['extra']}"

            dict_vector = {}

            for vector_name, model_name in models_names.items():
                dict_vector[vector_name] = models.Document(
                    text = text_embed,
                    model = model_name
                )

            point = models.PointStruct(
                id = record['id'],
                vector = dict_vector,
                payload = {
                    'term': record['term'],
                    'description': f"{record['definition']} {record['extra']}",
                    'models_used': models_names
                }
            )

            points.append(point)

        client.upsert(
            collection_name = name,
            points=points
        )

        print(f"Successfully populated collection '{name}' with {len(points)} records.")

    except Exception as e:
        print(f"An error occurred: {e}")

### Hybrid-reranking search

In [55]:
def rrf_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):
        print(f"Collection '{collection_name}' does not exist.")
        return

    collection = client.get_collection(collection_name)
    vector_config = collection.config.params.vectors
    sparse_vector_config = collection.config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 2:
        print("Both dense and sparse vectors are required for RRF search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]
    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():
        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:
        print("Dense and sparse vector fields not found in the collection.")
        return

    results = client.query_points(
        collection_name=collection_name,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=dense["model"]
                ),
                using=dense["name"],
                limit=2 * limit
            ),
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=sparse["model"]
                ),
                using=sparse["name"],
                limit=2 * limit
            )
        ],
        limit=limit,
        with_payload=True
    )

    context = {'question': question}

    for i, res in enumerate(results.points):
        context[f'context{i+1}'] = res.payload['description']

    return context

In [9]:
build_collection(
    name = 'hybrid-search-collection',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    },
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='hybrid-search-collection',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
        'sparse_text': 'Qdrant/bm25'
    },
    documents=documents
)

Qdrant collection 'hybrid-search-collection' created.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

romanian.txt: 0.00B [00:00, ?B/s]

spanish.txt: 0.00B [00:00, ?B/s]

swedish.txt:   0%|          | 0.00/559 [00:00<?, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]

Successfully populated collection 'hybrid-search-collection' with 829 records.


### Input prompt for Qdrant collection for search

In [95]:
input_prompt = """

    You will be given:
    1. A question.
    2. Five context paragraphs (labeled context1 through context5).

    Your task:
    - Answer the question **using only the information provided in the contexts**.
    - Do not use outside knowledge.
    - If the contexts do not contain enough information to fully answer, say so explicitly.
    - Make the answer clear, concise, and directly related to the question.

    Input format:

    question: {question}
    context1: {context1}
    context2: {context2}
    context3: {context3}
    context4: {context4}
    context5: {context5}

""".strip()

### Get relevant documents from collection and generate response using LLM

In [113]:
def get_response(question: str, model: str = 'gemini-2.5-flash-lite'):

    context_docs = rrf_search(question=question, collection_name='hybrid-search-collection')
    prompt = input_prompt.format(**context_docs)

    response = google_client.models.generate_content(
        model = model,
        contents = prompt
    )

    return response.text

### Evaluation prompt for response

In [131]:
prompt_evaluate = """

    You will be given:
    1. A question.
    2. A generated answer.

    Your task:
    - Evaluate how relevant the answer is to the question.
    - Use only the information in the given input.

    Output requirements:
    - Assign a relevance score: RELEVANT | PARTIALLY RELEVANT | NOT RELEVANT
    - Provide a brief explanation for your choice.
    - Output must be plain text only, not a code block or JSON block.
    - Format your output exactly as shown, using plain text and single quotes:

    {{'Relevance': '<your score>', 'Explanation': '<your explanation>'}}

    Input:

    question: {question}
    answer: {answer}

""".strip()


### Evaluate generated response with LLM

In [132]:
def get_evaluation(question: str, answer: str):

    dictionary = {'question': question, 'answer': answer}
    prompt = prompt_evaluate.format(**dictionary)

    response = google_client.models.generate_content(
        model = 'gemma-3n-e2b-it',
        contents = prompt
    )

    return response.text

In [112]:
question = "What am doing here?"

In [13]:
random_row = questions.sample(n=1)

In [56]:
docs = rrf_search(question=random_row.iloc[0]['question'], collection_name='hybrid-search-collection')

In [116]:
answer = get_response(question = question)

In [117]:
print(answer)

The provided contexts do not contain information about what "I" am doing here.


In [133]:
print(get_evaluation(question=question, answer=answer))

{'Relevance': 'NOT RELEVANT', 'Explanation': 'The question asks what the user is doing, but the answer states that the context doesn\'t provide information about it. Therefore, the answer is not relevant to the question.'}


In [69]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

In [71]:
google_client = genai.Client(api_key=GOOGLE_API_KEY)

In [73]:
response = google_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=input,
)

print(response.text)

The term 'video' has broadened its meaning to cover everything from 'streaming content' to social media clips, beyond its original use for films or television programs recorded digitally or on tape.


The term 'video' has broadened its meaning to cover everything from 'streaming content' to social media clips, beyond its original use for films or television programs recorded digitally or on tape.

In [134]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=51,
  prompt_token_count=195,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=195
    ),
  ],
  total_token_count=246
)

In [75]:
response = google_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Is this friends?",
)

print(response.text)

To answer that, I need more information! What are you referring to when you say "this"?

Please tell me:

*   **What is "this"?** Is it a picture, a situation you're describing, a conversation you're having, or something else?
*   **What makes you ask if it's friends?** What are you observing or experiencing that makes you wonder?

Once you provide me with more context, I can help you figure it out!


In [76]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=101,
  prompt_token_count=5,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=5
    ),
  ],
  total_token_count=106
)

In [80]:
answer = "The term 'video' has broadened its meaning to cover everything from 'streaming content' to social media clips, beyond its original use for films or television programs recorded digitally or on tape."

In [81]:
evaluate_dict = {'question': random_row.iloc[0]['question'], 'answer': answer}

In [82]:
evaluate_dict

{'question': "Can you give an example of how the term 'video' has broadened its meaning over time?",
 'answer': "The term 'video' has broadened its meaning to cover everything from 'streaming content' to social media clips, beyond its original use for films or television programs recorded digitally or on tape."}

In [85]:
output = prompt_evaluate.format(**evaluate_dict)

In [87]:
response = google_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=output,
)

print(response.text)

{'Relevance': 'RELEVANT', 'Explanation': "The answer directly addresses the question by providing examples of how the term 'video' has broadened its meaning, citing 'streaming content' and social media clips as examples beyond its original uses."}


In [135]:
response.usage_metadata.prompt_token_count

195